In [28]:
import pandas as pd
import numpy as np

In [29]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [30]:
reserved_df = pd.read_csv("raw_data/reserved.csv")
reserved_df.head()

,N-NUMBER,REGISTRANT,STREET,STREET2,CITY,STATE,ZIP CODE,RSV DATE,TR,EXP DATE,N-NUM-CHG,PURGE DATE,Unnamed: 12
0,1,FAA,NATL FLIGHT PROGRAM OVERSIGHT,6125 SW 68TH ST RM 137N,OKLAHOMA CITY,OK,73169-6972,20240930,AA,,,,NaN
1,10,FEDERAL AVIATION ADMINISTRATION,NATL FLIGHT PROGRAM OVERSIGHT OFC,6125 SW 68TH STREET ROOM 137N,OKLAHOMA CITY,OK,73169,19821202,AA,,,,NaN
2,1000,MAHONEY THOMAS M JR,PO BOX 9,,OCOEE,FL,34761-0009,20240205,A,20251205,,20260305,NaN
3,10002,CANCELLED/NOT ASSIGNED,,,,,,20210130,HD,,,20260130,NaN
4,10003,CANCELLED/NOT ASSIGNED,,,,,,20210703,HD,,,20260703,NaN


In [31]:
reserved_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 126591 entries, 0 to 126590
Data columns (total 13 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   N-NUMBER     126591 non-null  str    
 1   REGISTRANT   126591 non-null  str    
 2   STREET       126591 non-null  str    
 3   STREET2      126591 non-null  str    
 4   CITY         126591 non-null  str    
 5   STATE        126591 non-null  str    
 6   ZIP CODE     126591 non-null  str    
 7   RSV DATE     126591 non-null  int64  
 8   TR           126591 non-null  str    
 9   EXP DATE     126591 non-null  str    
 10  N-NUM-CHG    126591 non-null  str    
 11  PURGE DATE   126591 non-null  str    
 12  Unnamed: 12  0 non-null       float64
dtypes: float64(1), int64(1), str(11)
memory usage: 12.6 MB


In [32]:
reserved_df.columns = reserved_df.columns.str.strip()

df_obj_cols = reserved_df.select_dtypes(['object']).columns

reserved_df[df_obj_cols] = reserved_df[df_obj_cols].apply(lambda x: x.astype(str).str.strip())

if 'Unnamed: 12' in reserved_df.columns:
    reserved_df = reserved_df.drop(columns=['Unnamed: 12'])

reserved_df.head()

/var/folders/r5/7sdqfmvs4p987n2g0n_nng840000gn/T/ipykernel_54849/2904168487.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df_obj_cols = reserved_df.select_dtypes(['object']).columns


,N-NUMBER,REGISTRANT,STREET,STREET2,CITY,STATE,ZIP CODE,RSV DATE,TR,EXP DATE,N-NUM-CHG,PURGE DATE
0,1,FAA,NATL FLIGHT PROGRAM OVERSIGHT,6125 SW 68TH ST RM 137N,OKLAHOMA CITY,OK,73169-6972,20240930,AA,,,
1,10,FEDERAL AVIATION ADMINISTRATION,NATL FLIGHT PROGRAM OVERSIGHT OFC,6125 SW 68TH STREET ROOM 137N,OKLAHOMA CITY,OK,73169,19821202,AA,,,
2,1000,MAHONEY THOMAS M JR,PO BOX 9,,OCOEE,FL,34761-0009,20240205,A,20251205,,20260305
3,10002,CANCELLED/NOT ASSIGNED,,,,,,20210130,HD,,,20260130
4,10003,CANCELLED/NOT ASSIGNED,,,,,,20210703,HD,,,20260703


In [33]:
cols_to_fix = reserved_df.columns

reserved_df[df_obj_cols] = reserved_df[df_obj_cols].replace(r'^\s*$', np.nan, regex=True)

In [34]:
date_cols = ['EXP DATE', 'PURGE DATE']
for col in date_cols:
    if col in reserved_df.columns:
        reserved_df[col] = pd.to_datetime(reserved_df[col], format='%Y%m%d', errors='coerce')

In [35]:
reserved_df

,N-NUMBER,REGISTRANT,STREET,STREET2,CITY,STATE,ZIP CODE,RSV DATE,TR,EXP DATE,N-NUM-CHG,PURGE DATE
0,1,FAA,NATL FLIGHT PROGRAM OVERSIGHT,6125 SW 68TH ST RM 137N,OKLAHOMA CITY,OK,73169-6972,20240930,AA,NaT,NaN,NaT
1,10,FEDERAL AVIATION ADMINISTRATION,NATL FLIGHT PROGRAM OVERSIGHT OFC,6125 SW 68TH STREET ROOM 137N,OKLAHOMA CITY,OK,73169,19821202,AA,NaT,NaN,NaT
2,1000,MAHONEY THOMAS M JR,PO BOX 9,NaN,OCOEE,FL,34761-0009,20240205,A,2025-12-05,NaN,2026-03-05
3,10002,CANCELLED/NOT ASSIGNED,NaN,NaN,NaN,NaN,NaN,20210130,HD,NaT,NaN,2026-01-30
4,10003,CANCELLED/NOT ASSIGNED,NaN,NaN,NaN,NaN,NaN,20210703,HD,NaT,NaN,2026-07-03
...,...,...,...,...,...,...,...,...,...,...,...,...
126586,9ZM,SHORT-N-NUMBERS,PO BOX 368,NaN,GROVELAND,FL,34736-0368,20170131,FP,2026-11-30,NaN,2027-02-28
126587,9ZN,SHORT-N-NUMBERS,PO BOX 368,NaN,GROVELAND,FL,34736-0368,20161220,FP,2026-10-20,NaN,2027-01-20
126588,9ZW,SHORT-N-NUMBERS,PO BOX 368,NaN,GROVELAND,FL,34736-0368,20150522,FP,2026-03-22,NaN,2026-06-22
126589,9ZY,SHORT-N-NUMBERS,PO BOX 368,NaN,GROVELAND,FL,34736-0368,20161011,FP,2026-08-11,NaN,2026-11-11


## Cleaning `N-NUMBER` column
- Prepend all registrations with "N" (since they're all reserved from the FAA in the USA)

In [36]:
reserved_df['N-NUMBER'] = 'N' + reserved_df['N-NUMBER'].astype(str)
reserved_df['N-NUM-CHG'] = 'N' + reserved_df['N-NUM-CHG'].astype(str)

In [37]:
reserved_df["N-NUMBER"].is_unique

True

In [38]:
reserved_df["N-NUMBER"]

0             N1
1            N10
2          N1000
3         N10002
4         N10003
           ...  
126586      N9ZM
126587      N9ZN
126588      N9ZW
126589      N9ZY
126590      N9ZZ
Name: N-NUMBER, Length: 126591, dtype: str

## Cleaning `TR` column

From the FAA dataset description, we know:

* AA - Reserved - no fee
* A - Fee paid, notice for expiration sent
* HD - 2 year hold for canceled N-Numbers
* FN - Fee paid, notice for expiration sent
* FP - Fee paid
* MF - Reserved to manufacturer - no fee, no expiration date
* MT - Reserved to manufacturer - temporary number
* NC - N-Number change is in process
* NN - N-Number change is in process,
* expiration notice sent
* CN - N-Number change, Expire Notice Sent
* CE – N-Number change Expired

In [39]:
reserved_df['TR'].value_counts(dropna=False)

TR
AA    57308
FP    36975
HD    23798
A      4831
MF     1313
MT     1302
NC      917
NN       70
CE       50
CN       27
Name: count, dtype: int64

In [40]:
tr_mapping = {
    "AA": "Reserved - no fee",
    "A": "Fee paid, notice for expiration sent",
    "HD": "2 year hold for canceled N-Numbers",
    "FN": "Fee paid, notice for expiration sent",
    "FP": "Fee paid",
    "MF": "Reserved to manufacturer - no fee, no expiration date",
    "MT": "Reserved to manufacturer - temporary number",
    "NC": "N-Number change is in process",
    "NN": "N-Number change is in process, expiration notice sent",
    "CN": "N-Number change, Expire Notice Sent",
    "CE": "N-Number change Expired"
}

reserved_df['TR'] = reserved_df['TR'].map(tr_mapping)

In [41]:
reserved_df['TR'].value_counts(dropna=False)

TR
Reserved - no fee                                        57308
Fee paid                                                 36975
2 year hold for canceled N-Numbers                       23798
Fee paid, notice for expiration sent                      4831
Reserved to manufacturer - no fee, no expiration date     1313
Reserved to manufacturer - temporary number               1302
N-Number change is in process                              917
N-Number change is in process, expiration notice sent       70
N-Number change Expired                                     50
N-Number change, Expire Notice Sent                         27
Name: count, dtype: int64

## Column Name Cleaning

In [46]:
reserved_df.columns = (reserved_df.columns
                     .str.replace(' ', '_', regex=False)
                     .str.replace('-', '_', regex=False)
                     .str.replace('(', '', regex=False)
                     .str.replace(')', '', regex=False)
                     .str.lower())

In [47]:
reserved_df.to_csv('./clean_data/RESERVED.csv', index=False, na_rep='NULL')